# CSC2042S — Assignment 4: Neural Networks (Submission-ready)

**Submission notes:**

This notebook is a clean, submission-ready implementation of the assignment. All experiment results, tables, and figures are presented inline — no external CSV or model checkpoint files are created or required. When you run the notebook cells it will perform the training and display results inside the notebook.

**Important:** Running the baseline (10 epochs) and hyperparameter sweep (several trainings) may take a noticeable amount of time. Use a GPU-enabled environment if available.


---

## Quick setup

Install required packages if missing:

```bash
pip install torch torchvision matplotlib scikit-learn pandas seaborn notebook
```

The notebook uses `torchvision.datasets.FashionMNIST(download=True)` by default for simplicity.

---


In [ ]:

# Imports and reproducibility
import os, time, copy, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## Task 1 — Data processing

In [ ]:

# === Task 1: Data processing ===
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

def get_dataloaders(batch_size=128, val_split=0.1, data_dir='./data'):
    train_full = datasets.FashionMNIST(root=data_dir, train=True, download=True, transform=transform)
    test = datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=transform)
    n_val = int(len(train_full) * val_split)
    n_train = len(train_full) - n_val
    train_ds, val_ds = random_split(train_full, [n_train, n_val])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

# Sanity check (light)
train_loader, val_loader, test_loader = get_dataloaders(batch_size=128)
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader), 'Test batches:', len(test_loader))


## Task 2 — Baseline model and training

In [ ]:

# === Task 2: Model definition and training utilities ===
class MLP(nn.Module):
    def __init__(self, input_size=28*28, hidden1=128, hidden2=64, num_classes=10, activation=nn.ReLU, dropout_p=0.0):
        super().__init__()
        layers = [nn.Flatten(),
                  nn.Linear(input_size, hidden1),
                  activation()]
        if dropout_p>0: layers.append(nn.Dropout(dropout_p))
        layers += [nn.Linear(hidden1, hidden2), activation()]
        if dropout_p>0: layers.append(nn.Dropout(dropout_p))
        layers.append(nn.Linear(hidden2, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0; correct=0; total=0
    for xb,yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*xb.size(0)
        _, preds = out.max(1)
        correct += (preds==yb).sum().item()
        total += yb.size(0)
    return running_loss/total, correct/total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss=0.0; correct=0; total=0
    all_preds=[]; all_targets=[]
    with torch.no_grad():
        for xb,yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            running_loss += loss.item()*xb.size(0)
            _, preds = out.max(1)
            correct += (preds==yb).sum().item()
            total += yb.size(0)
            all_preds.append(preds.cpu().numpy()); all_targets.append(yb.cpu().numpy())
    if len(all_preds)>0:
        all_preds = np.concatenate(all_preds); all_targets = np.concatenate(all_targets)
    else:
        all_preds = np.array([]); all_targets = np.array([])
    return running_loss/total, correct/total, all_preds, all_targets

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=10, scheduler=None):
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_acc = 0.0; best_model_wts = copy.deepcopy(model.state_dict())
    for e in range(epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        if scheduler is not None:
            try: scheduler.step(val_loss)
            except: scheduler.step()
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
        print(f"Epoch {e+1}/{epochs} train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} time={(time.time()-t0):.1f}s")
    model.load_state_dict(best_model_wts)
    return model, history


In [ ]:

# === Baseline run (Task 2) ===
set_seed(42)
batch_size = 128
train_loader, val_loader, test_loader = get_dataloaders(batch_size=batch_size, val_split=0.1)
baseline_model = MLP(hidden1=128, hidden2=64, activation=nn.ReLU, dropout_p=0.0).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)
baseline_epochs = 10  # as per spec
print('Training baseline model for', baseline_epochs, 'epochs (this may take several minutes if running).')
baseline_model, baseline_history = train_model(baseline_model, train_loader, val_loader, criterion, optimizer, device, epochs=baseline_epochs)
val_loss, val_acc, val_preds, val_targets = evaluate(baseline_model, val_loader, criterion, device)
print(f'Baseline final validation accuracy: {val_acc:.4f}')
# Display baseline epoch table neatly
import pandas as _pd
baseline_epoch_df = _pd.DataFrame({
    'epoch': list(range(1, len(baseline_history['train_acc'])+1)),
    'train_acc': baseline_history['train_acc'],
    'val_acc': baseline_history['val_acc'],
    'train_loss': baseline_history['train_loss'],
    'val_loss': baseline_history['val_loss'],
})
display(baseline_epoch_df.style.format({'train_acc':'{:.4f}','val_acc':'{:.4f}','train_loss':'{:.4f}','val_loss':'{:.4f}'}))


## Task 3 — Hyperparameter experiments (grid search)

In [ ]:

# === Hyperparameter sweep (Task 3) ===
learning_rates = [0.1, 0.01, 0.001]
optimizers = ['SGD','Adam','RMSprop']
dropouts = [0.0, 0.25, 0.5]

def make_optimizer(name, params, lr):
    if name=='SGD': return optim.SGD(params, lr=lr, momentum=0.9)
    if name=='Adam': return optim.Adam(params, lr=lr)
    if name=='RMSprop': return optim.RMSprop(params, lr=lr)
    raise ValueError(name)

results = []
best_overall = {'val_acc':0.0}
max_epochs = 8  # reduced for sweep speed; bump this if you want more thorough search
set_seed(42)
for lr in learning_rates:
    for opt_name in optimizers:
        for dp in dropouts:
            cfg = {'lr':lr,'optimizer':opt_name,'dropout':dp}
            print('Running', cfg)
            set_seed(42)
            model = MLP(hidden1=128, hidden2=64, activation=nn.ReLU, dropout_p=dp).to(device)
            criterion = nn.CrossEntropyLoss()
            optimizer = make_optimizer(opt_name, model.parameters(), lr)
            model, history = train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=max_epochs)
            val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
            results.append({'lr':lr,'optimizer':opt_name,'dropout':dp,'val_acc':val_acc,'val_loss':val_loss,'history':history,'model':model})
            if val_acc > best_overall['val_acc']:
                best_overall = {'val_acc':val_acc,'lr':lr,'optimizer':opt_name,'dropout':dp,'history':history,'model':copy.deepcopy(model)}
            print(f"Config result val_acc={val_acc:.4f}\n")

# Build results DataFrame for inline presentation (no file saving)
results_df = pd.DataFrame([{'lr':r['lr'],'optimizer':r['optimizer'],'dropout':r['dropout'],'val_acc':r['val_acc'],'val_loss':r['val_loss']} for r in results])
results_df = results_df.sort_values('val_acc', ascending=False).reset_index(drop=True)

# Expose best model and history for analysis
best_trained_model = best_overall['model']
best_history = best_overall['history']

print('Best configuration (inline):')
print({'lr':best_overall['lr'],'optimizer':best_overall['optimizer'],'dropout':best_overall['dropout'],'val_acc':best_overall['val_acc']})
display(results_df.style.format({'val_acc':'{:.4f}','val_loss':'{:.4f}'}))


## Task 4 — Results presentation & analysis (all inline, no external files)

In [ ]:

# === Analysis & Visualisations (Task 4) ===
# 1) Ranked table (already created by sweep) and highlight best config
try:
    display(results_df.style.format({'val_acc':'{:.4f}','val_loss':'{:.4f}'}))
    best_cfg = {'lr':best_overall['lr'],'optimizer':best_overall['optimizer'],'dropout':best_overall['dropout'],'val_acc':best_overall['val_acc']}
    from IPython.display import Markdown, display as _display
    _display(Markdown('**Best configuration found (highlighted):**'))
    _display(Markdown(f"- learning rate: **{best_cfg['lr']}**  \n- optimizer: **{best_cfg['optimizer']}**  \n- dropout: **{best_cfg['dropout']}**  \n- validation accuracy: **{best_cfg['val_acc']:.4f}**"))
except NameError:
    print('Run the hyperparameter sweep cell first to generate results_df and best_trained_model.')

# 2) Heatmaps: impact of lr vs dropout for each optimizer
if 'results_df' in globals():
    for opt in results_df['optimizer'].unique():
        sub = results_df[results_df['optimizer']==opt]
        pivot = sub.pivot(index='dropout', columns='lr', values='val_acc')
        plt.figure(figsize=(6,4))
        sns.heatmap(pivot, annot=True, fmt='.4f', cbar_kws={'label':'val_acc'})
        plt.title(f'Validation Accuracy heatmap (optimizer={opt})')
        plt.ylabel('dropout'); plt.xlabel('learning rate')
        plt.suptitle('')  # suppress extra suptitle in some environments
        plt.show()
        display(Markdown(f"_Figure:_ Heatmap of validation accuracy for optimizer **{opt}**, rows=dropout, columns=learning rate"))

# 3) Loss curves: baseline vs best
try:
    baseline_hist = baseline_history
    bh = best_history
    plt.figure(figsize=(8,5))
    plt.plot(baseline_hist['val_loss'], label='Baseline val_loss', linestyle='--')
    plt.plot(baseline_hist['train_loss'], label='Baseline train_loss', linestyle=':')
    plt.plot(bh['val_loss'], label='Best val_loss')
    plt.plot(bh['train_loss'], label='Best train_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss curves: Baseline vs Best'); plt.legend(); plt.show()
    display(Markdown("_Figure:_ Loss curves showing baseline vs best model (per-epoch)."))
except Exception as e:
    print('Could not plot loss curves; ensure baseline and sweep have been run.', e)

# 4) Confusion matrix for best model on validation set
try:
    bm = best_trained_model
    _, _, preds, targets = evaluate(bm, val_loader, criterion, device)
    cm = confusion_matrix(targets, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=[str(i) for i in range(10)])
    fig, ax = plt.subplots(figsize=(8,8))
    disp.plot(ax=ax, cmap='viridis', xticks_rotation='vertical')
    plt.title('Confusion Matrix - Best Model on Validation Set'); plt.show()
    display(Markdown("_Figure:_ Confusion matrix for the best model on the validation set. Use this for error analysis."))
except Exception as e:
    print('Could not compute confusion matrix automatically:', e)

# 5) Epoch accuracy tables side-by-side
try:
    import pandas as _pd
    baseline_df = _pd.DataFrame({'epoch': list(range(1,len(baseline_history['train_acc'])+1)),
                                 'train_acc': baseline_history['train_acc'],
                                 'val_acc': baseline_history['val_acc']})
    best_df = _pd.DataFrame({'epoch': list(range(1,len(best_history['train_acc'])+1)),
                             'train_acc': best_history['train_acc'],
                             'val_acc': best_history['val_acc']})
    display(Markdown("**Baseline epoch accuracies**"))
    display(baseline_df.style.format({'train_acc':'{:.4f}','val_acc':'{:.4f}'}))
    display(Markdown("**Best model epoch accuracies**"))
    display(best_df.style.format({'train_acc':'{:.4f}','val_acc':'{:.4f}'}))
except Exception as e:
    print('Could not display epoch tables:', e)

# 6) Short interpretation helper (one-liners under each figure/table)
display(Markdown("**Interpretation pointers (short):**\n- Compare baseline final val_acc with the best config val_acc and comment on improvement.\n- Use heatmaps to see which hyperparameter changes produced the largest swings in accuracy.\n- Use confusion matrix to find two commonly confused classes and suggest reasons (e.g., visual similarity between classes).\n"))


---
## Optional Kaggle instructions
If you prefer to download the dataset using Kaggle, use the Kaggle CLI. Otherwise torchvision will download automatically.
---


---
## Report guidance
Use the inline tables and figures in this notebook to prepare your report. Keep the report to a maximum of 5 pages, addressing:
- Baseline vs Best: final accuracies and percent improvement.
- Hyperparameter influence: which hyperparameter mattered most and why.
- Overfitting analysis: examine the loss curves.
- Error analysis: two confused classes and a plausible explanation.
---
